# Urban Heat & Cooling-Priority Mapping — NEA Held-Out Validation

**NUS-ISS Practice Module, Week 2.** Builds `nea_heldout_lst.csv` for
`rank_impact.ipynb`'s `lst_rmse_heldout` column, from the same NEA
air-temperature API your Week-1 G2 gate already validated.

**⚠️ Read before trusting the output:** NEA stations measure *air*
temperature (~1.5m, shaded instruments), not *land surface* temperature
(the satellite thermal band your heat-layer variants are built from). LST
typically runs several degrees higher than air temp over built-up surfaces,
especially near satellite overpass time. This is the best independent,
zero-cost validation source available in scope — but it's a proxy with a
systematic offset baked in, not a clean ground truth. State that plainly as
a limitation; don't present RMSE against this as if it were validated
against true LST. Spearman/top-20 overlap in `rank_impact.ipynb` are less
sensitive to this since they're rank-based, not magnitude-based.

Coverage will also be inherently sparse: Singapore has on the order of a few
dozen weather stations against ~330 subzones, so most subzones will have no
matched station and won't appear in the output at all.

One notebook, run top to bottom:

1. **Setup** — install deps, mount Drive (no Earth Engine needed)
2. **NH.1** — config
3. **NH.2** — fetch URA subzones (same source as `gee_heat_variants.ipynb`)
4. **NH.3** — fetch NEA station metadata
5. **NH.4** — spatial join: assign each station to a subzone
6. **NH.5** — fetch temperature readings across the season window (sampled days)
7. **NH.6** — aggregate to per-subzone `lst_heldout_c`
8. **NH.7** — match subzone_id to heat-variants CSV
9. **NH.8** — verdict
10. **NH.9** — save to Drive

Run cells in order.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q requests pandas geopandas shapely


## Setup 2 — Mount Google Drive

In [2]:
# --- SETUP CELL 2: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 3 — Initialize shared results tracker

In [3]:
# --- SETUP CELL 3: Initialize shared results tracker ------------------------
nh_results = {}
print("nh_results initialized — populated by the NH.8 verdict cell.")


nh_results initialized — populated by the NH.8 verdict cell.


---
# NH — NEA Held-Out Validation (air-temperature proxy for LST)


## NH.1 — Config

`YEARS` / `DRY_SEASON_MONTHS` should match `gee_heat_variants.ipynb` so the
validation window is at least seasonally consistent with the composites
being validated — it can't be pixel-date-matched since NEA readings are
current/near-real-time only, not retrievable for arbitrary past composite
dates going back multiple years on the free tier used here.


In [4]:
# --- NH CELL 1: Config -------------------------------------------------------
NEA_API_KEY = ""  # optional — data.gov.sg's real-time APIs work without a key
                   # until rate-limited (HTTP 429); get one at https://data.gov.sg
                   # (sign in -> Create API Key -> Developer) if you hit that.
NEA_BASE_URL = "https://api-open.data.gov.sg/v2/real-time/api/air-temperature"

# Sample N days spread across the last N_MONTHS_BACK months as a practical stand-in
# for "the season window" — the free real-time API doesn't expose arbitrary
# multi-year historical dates the way the GEE composites can reach back to 2021.
N_SAMPLE_DAYS = 20
N_MONTHS_BACK = 6
SAMPLE_SEED = 42  # pinned, per eval rules

EXPORT_FOLDER = "urban_heat_sg"
HEAT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/heat_variants_subzone.csv"
OUT_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/nea_heldout_lst.csv"

SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"  # MP19 Subzone Boundary (No Sea), GEOJSON
SUBZONE_LOCAL_PATH = "/content/ura_subzones_nh.geojson"
SUBZONE_ID_PROPERTY = "SUBZONE_N"

import os
if not os.path.exists(HEAT_CSV_PATH):
    raise FileNotFoundError(
        f"{HEAT_CSV_PATH} not found. Run gee_heat_variants.ipynb's export first, "
        f"then re-run this cell."
    )
print("✅ Heat-variants CSV found — will match subzone_id against it in NH.7.")


✅ Heat-variants CSV found — will match subzone_id against it in NH.7.


## NH.2 — Fetch URA subzones

Loaded with `geopandas` this time, not `ee.FeatureCollection` — no Earth
Engine needed for a client-side point-in-polygon join, so this stays
independent of EE auth/quota entirely.


In [5]:
# --- NH CELL 2: Fetch URA subzones (geopandas, not EE) -----------------------
import requests
import geopandas as gpd

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    download_url = payload["data"]["url"]
    geojson_bytes = requests.get(download_url).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, SUBZONE_LOCAL_PATH)
subzones_gdf = gpd.read_file(subzone_path)
print(f"Loaded {len(subzones_gdf)} subzones. CRS: {subzones_gdf.crs}")

if SUBZONE_ID_PROPERTY not in subzones_gdf.columns:
    candidates = [c for c in subzones_gdf.columns if "name" in c.lower() or "subzone" in c.lower()]
    raise ValueError(f"'{SUBZONE_ID_PROPERTY}' not found. Candidates: {candidates}")
print(f"✅ SUBZONE_ID_PROPERTY = '{SUBZONE_ID_PROPERTY}' confirmed present.")

if subzones_gdf.crs is None or subzones_gdf.crs.to_epsg() != 4326:
    print(f"Reprojecting subzones from {subzones_gdf.crs} to EPSG:4326 to match station lat/lon...")
    subzones_gdf = subzones_gdf.to_crs(epsg=4326)


Loaded 332 subzones. CRS: EPSG:4326
✅ SUBZONE_ID_PROPERTY = 'SUBZONE_N' confirmed present.


## NH.3 — Fetch NEA station metadata

In [6]:
# --- NH CELL 3: Fetch NEA station metadata -----------------------------------
import pandas as pd

headers = {"x-api-key": NEA_API_KEY} if NEA_API_KEY else {}

def fetch_air_temp(date_str=None):
    params = {"date": date_str} if date_str else {}
    r = requests.get(NEA_BASE_URL, params=params, headers=headers)
    return r

resp = fetch_air_temp()
print("HTTP status:", resp.status_code)

if resp.status_code == 429:
    print("⚠️  Rate-limited. Get a free key at https://data.gov.sg (sign in -> Create API Key -> Developer),")
    print("   set NEA_API_KEY in NH.1, then re-run from NH.3.")
    raise RuntimeError("Rate-limited — see message above.")
resp.raise_for_status()

latest_json = resp.json()
station_metadata = latest_json.get("data", {}).get("stations", [])
n_stations = len(station_metadata)
print(f"Stations reported: {n_stations}")
if n_stations == 0:
    raise RuntimeError("Zero stations returned — API response shape may have changed; inspect latest_json.")

try:
    stations_df = pd.DataFrame([
        {
            "station_id": s["id"],
            "station_name": s.get("name", ""),
            "latitude": s["location"]["latitude"],
            "longitude": s["location"]["longitude"],
        }
        for s in station_metadata
    ])
except KeyError as e:
    print(f"⚠️  Expected field {e} not found in station metadata. Actual shape of first station record:")
    print(station_metadata[0])
    print("Adjust the field names in this cell to match, then re-run.")
    raise
print(stations_df.head())


HTTP status: 200
Stations reported: 16
  station_id         station_name  latitude  longitude
0       S109  Ang Mo Kio Avenue 5    1.3793   103.8500
1       S106           Pulau Ubin    1.4168   103.9673
2       S117          Banyan Road    1.2542   103.6741
3       S107   East Coast Parkway    1.3133   103.9620
4       S104   Woodlands Avenue 9    1.4439   103.7854


## NH.4 — Spatial join: assign each station to a subzone

Point-in-polygon. Stations outside every subzone polygon (e.g. offshore
buoys, Sentosa/reservoir edge cases) are expected and dropped, not an error.


In [7]:
# --- NH CELL 4: Spatial join (stations -> subzones) --------------------------
from shapely.geometry import Point

stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=[Point(xy) for xy in zip(stations_df["longitude"], stations_df["latitude"])],
    crs="EPSG:4326",
)

joined = gpd.sjoin(stations_gdf, subzones_gdf[[SUBZONE_ID_PROPERTY, "geometry"]], how="left", predicate="within")
joined = joined.rename(columns={SUBZONE_ID_PROPERTY: "subzone_id"})

n_matched_stations = joined["subzone_id"].notna().sum()
n_unmatched_stations = joined["subzone_id"].isna().sum()
n_subzones_with_station = joined["subzone_id"].nunique()

print(f"Stations matched to a subzone: {n_matched_stations} / {len(joined)}")
print(f"Stations outside all subzone polygons (dropped): {n_unmatched_stations}")
print(f"Distinct subzones with >=1 station: {n_subzones_with_station} / {len(subzones_gdf)} "
      f"({100*n_subzones_with_station/len(subzones_gdf):.1f}%)")
print("ℹ️  Low coverage is expected given station count vs subzone count — not a bug.")

station_to_subzone = joined.dropna(subset=["subzone_id"])[["station_id", "subzone_id"]].set_index("station_id")["subzone_id"].to_dict()


Stations matched to a subzone: 15 / 16
Stations outside all subzone polygons (dropped): 1
Distinct subzones with >=1 station: 14 / 332 (4.2%)
ℹ️  Low coverage is expected given station count vs subzone count — not a bug.


## NH.5 — Fetch temperature readings across sampled days

Samples `N_SAMPLE_DAYS` dates spread over the last `N_MONTHS_BACK` months
(seeded, per eval rules) rather than every single day, to keep this to a
reasonable number of API calls. Each call returns all readings for that day
(commonly multiple timestamps); all are pooled before averaging.


In [8]:
# --- NH CELL 5: Fetch readings across sampled days ----------------------------
import random
import time
from datetime import datetime, timedelta

random.seed(SAMPLE_SEED)
today = datetime.now()
candidate_days = [(today - timedelta(days=d)).strftime("%Y-%m-%d") for d in range(1, N_MONTHS_BACK * 30)]
sample_days = sorted(random.sample(candidate_days, min(N_SAMPLE_DAYS, len(candidate_days))))
print(f"Sampling {len(sample_days)} days: {sample_days[0]} .. {sample_days[-1]}")

all_readings = []  # list of (station_id, value)
n_days_ok = 0
schema_error_shown = False
for d in sample_days:
    r = fetch_air_temp(date_str=d)
    if r.status_code != 200:
        print(f"  {d}: HTTP {r.status_code}, skipped")
        time.sleep(0.3)
        continue
    payload = r.json()
    reading_batches = payload.get("data", {}).get("readings", [])
    try:
        for batch in reading_batches:
            for entry in batch.get("data", []):
                all_readings.append((entry["stationId"], entry["value"]))
    except KeyError as e:
        if not schema_error_shown:
            print(f"⚠️  Expected field {e} not found in reading entry. Actual shape of first batch:")
            print(reading_batches[0] if reading_batches else "(empty reading_batches)")
            print("Adjust the field names in this cell to match, then re-run.")
            schema_error_shown = True
        raise
    n_days_ok += 1
    time.sleep(0.3)  # be polite to the API

print(f"\nDays successfully fetched: {n_days_ok} / {len(sample_days)}")
print(f"Total (station, reading) pairs pooled: {len(all_readings)}")
if len(all_readings) == 0:
    raise RuntimeError("No readings collected — check API status/rate limiting before proceeding.")


Sampling 20 days: 2026-01-27 .. 2026-07-13

Days successfully fetched: 20 / 20
Total (station, reading) pairs pooled: 8198


## NH.6 — Aggregate to per-subzone `lst_heldout_c`

In [9]:
# --- NH CELL 6: Aggregate to per-subzone lst_heldout_c ------------------------
readings_df = pd.DataFrame(all_readings, columns=["station_id", "value"])
readings_df["subzone_id"] = readings_df["station_id"].map(station_to_subzone)

n_unmatched_readings = readings_df["subzone_id"].isna().sum()
readings_df = readings_df.dropna(subset=["subzone_id"])
print(f"Readings dropped (station not in any subzone): {n_unmatched_readings}")
print(f"Readings retained: {len(readings_df)}")

heldout_df = (
    readings_df.groupby("subzone_id")
    .agg(lst_heldout_c=("value", "mean"), n_readings=("value", "count"))
    .reset_index()
)
print(f"\nSubzones with a held-out value: {len(heldout_df)}")
print(heldout_df.sort_values('n_readings', ascending=False).head(10).to_string(index=False))


Readings dropped (station not in any subzone): 925
Readings retained: 7273

Subzones with a held-out value: 14
             subzone_id  lst_heldout_c  n_readings
                  MURAI      26.877333         825
             ANAK BUKIT      27.285000         500
JURONG ISLAND AND BUKOM      28.051200         500
         GREENWOOD PARK      27.261600         500
          NEWTON CIRCUS      27.259600         500
  NORTH-EASTERN ISLANDS      26.076400         500
    TUAS VIEW EXTENSION      27.978600         500
                SEMAKAU      28.553600         500
               TAI SENG      28.006800         500
                SENTOSA      27.921200         500


## NH.7 — Match subzone_id to heat-variants CSV

In [10]:
# --- NH CELL 7: Match subzone_id ----------------------------------------------
heat = pd.read_csv(HEAT_CSV_PATH)
heat_ids = set(heat["subzone_id"].astype(str))
nh_ids = set(heldout_df["subzone_id"].astype(str))

n_matched = len(heat_ids & nh_ids)
print(f"Matched: {n_matched} / {len(heldout_df)} held-out subzones also present in heat-variants CSV")

heldout_df = heldout_df[heldout_df["subzone_id"].astype(str).isin(heat_ids)].copy()
print(f"Final held-out table: {len(heldout_df)} subzones "
      f"({100*len(heldout_df)/len(heat_ids):.1f}% of all {len(heat_ids)} subzones in the heat-variants CSV)")


Matched: 14 / 14 held-out subzones also present in heat-variants CSV
Final held-out table: 14 subzones (4.2% of all 332 subzones in the heat-variants CSV)


## NH.8 — Verdict

In [11]:
# --- NH CELL 8: Verdict ---------------------------------------------------------
print("\n--- NH Verdict ---")

nh_checks = {
    "Station metadata fetched (non-zero)": n_stations > 0,
    "At least one day of readings fetched": n_days_ok > 0,
    "At least one subzone has a held-out value": len(heldout_df) > 0,
    "No majority-empty readings pool": len(readings_df) > 0,
}

for check, passed in nh_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

nh_pass = all(nh_checks.values())
print(f"\nCoverage: {len(heldout_df)} / {len(heat_ids)} subzones ({100*len(heldout_df)/len(heat_ids):.1f}%) "
      f"— low coverage is expected and NOT itself a failure; rank_impact.ipynb's RI.2 "
      f"already handles a partial-overlap held-out table.")
print("⚠️  Reminder: values are air temperature, a proxy for LST with a systematic offset. "
      "State this as a limitation, don't present RMSE against this as validated-against-true-LST.")

if nh_pass:
    print("\n✅ NH PASS: nea_heldout_lst.csv ready to save.")
else:
    print("\n⚠️  NH FAIL: resolve flagged step(s) above first.")

nh_results["NH_nea_heldout"] = {
    "status": "PASS" if nh_pass else "FAIL",
    "checks": nh_checks,
    "n_subzones_covered": len(heldout_df),
    "n_subzones_total": len(heat_ids),
    "coverage_pct": 100 * len(heldout_df) / len(heat_ids) if len(heat_ids) else 0,
}



--- NH Verdict ---
  [PASS] Station metadata fetched (non-zero)
  [PASS] At least one day of readings fetched
  [PASS] At least one subzone has a held-out value
  [PASS] No majority-empty readings pool

Coverage: 14 / 332 subzones (4.2%) — low coverage is expected and NOT itself a failure; rank_impact.ipynb's RI.2 already handles a partial-overlap held-out table.
⚠️  Reminder: values are air temperature, a proxy for LST with a systematic offset. State this as a limitation, don't present RMSE against this as validated-against-true-LST.

✅ NH PASS: nea_heldout_lst.csv ready to save.


## NH.9 — Save to Drive

In [12]:
# --- NH CELL 9: Save to Drive ---------------------------------------------------
out = heldout_df[["subzone_id", "lst_heldout_c"]]
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} ({len(out)} subzones)")
print("\nPoint HELDOUT_CSV_PATH at this file in rank_impact.ipynb's RI.1 to populate lst_rmse_heldout.")
print("Remember the air-temp-vs-LST caveat when writing this up.")


Saved: /content/drive/MyDrive/urban_heat_sg/nea_heldout_lst.csv (14 subzones)

Point HELDOUT_CSV_PATH at this file in rank_impact.ipynb's RI.1 to populate lst_rmse_heldout.
Remember the air-temp-vs-LST caveat when writing this up.
